In [ ]:
import requests
import pandas as pd
import numpy as np
from io import BytesIO
import matplotlib.pyplot as plt
import json
import os
import networkx as nx
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

from pathlib import Path

# 노트북을 어느 하위 폴더에서 실행해도 저장소 루트를 찾습니다.
_current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (_current_dir, *_current_dir.parents)
    if (path / "notebooks").is_dir() and (path / "data").is_dir()
)
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"


In [2]:
# DataFrame을 넣으면 모든 centrality를 계산해서 얻은 결과를 저장.
def cal_centrality(original_df):
    large_df = original_df[original_df['amount']!=0.0] # 토큰의 전송이 없는 transaction의 경우 제거
    # 네트워크 생성
    G = nx.from_pandas_edgelist(large_df, 'sender_address', 'receiver_address', edge_attr='amount', create_using=nx.DiGraph()) # 거래량으로 가중치를 부여.

    # Degree Centrality
    degree_centrality = nx.degree_centrality(G)
    
    # Betweenness Centrality
    betweenness_centrality = nx.betweenness_centrality(G)
    
    # Closeness Centrality
    closeness_centrality = nx.closeness_centrality(G)
    
    # Pagerank 감쇠계수는 보통 0.85
    pagerank = nx.pagerank(G, alpha=0.85) 

    centrality = {
    'degree_centrality' : degree_centrality, 
    'betweenness_centrality' : betweenness_centrality, 
    'closeness_centrality' : closeness_centrality, 
    'pagerank' : pagerank}

    df = pd.DataFrame(centrality)
    df.index.name = 'address'
    return df

In [3]:
# 메인 함수
def calculate_daily_centrality(df):
    # timestamp를 datetime 형식으로 변환
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # 날짜별로 그룹화
    grouped = df.groupby(df['timestamp'].dt.date)
    
    # 결과를 저장할 빈 DataFrame 생성
    result_df = pd.DataFrame()
    
    # tqdm을 사용하여 진행 상황 표시
    for date, group in tqdm(grouped):
        # 해당 날짜의 데이터에 대해 centrality 계산
        daily_centrality = cal_centrality(group)
        
        # 날짜 정보 추가
        daily_centrality['date'] = date
        
        # 결과를 메인 DataFrame에 추가
        result_df = pd.concat([result_df, daily_centrality], ignore_index=False)
    
    return result_df

In [ ]:
input_path = DATA_DIR / "processed" / "solana" / "SOL_2020-10-01_2020-12-31_TRX.csv"


In [ ]:
df = pd.read_csv(input_path)


In [ ]:
df.head()


In [20]:
# 함수 사용
result = calculate_daily_centrality(df)

100%|██████████████████████████████████████████████████████████████████████████████| 92/92 [2:24:27<00:00, 94.22s/it]


In [21]:
result

,degree_centrality,betweenness_centrality,closeness_centrality,pagerank,date
FrDcCb3DS3nFJ41jG3kvnsKuwjJf8qmWa89QcL4Jt4MA,0.000813,1.216617e-04,0.299410,0.000193,2020-10-01
8ZLcmKPxQ3151kNB5yXXYbFHnKLdkoLsEP4FtUWXgWkv,0.000813,4.135340e-08,0.199652,0.000196,2020-10-01
BzZeRXC4v3fqddMBYKLSP4Awswp5eSP3qUy9y9HsgXrG,0.000813,1.216617e-04,0.299410,0.000193,2020-10-01
6gfi6GSjrhqc5xDLtDkVrTR61Hi7GMNPmJknxvbqzb1x,1.196828,3.582936e-01,0.598414,0.277564,2020-10-01
6Mk9P4C77EskX51ov95z6RnFZ8jCpn7tbDVfeeTQbr28,0.000407,0.000000e+00,0.299309,0.000110,2020-10-01
...,...,...,...,...,...
5vZUJEfQK16qLT1q693LVVQPKuPTrFDM4zrS1oq8nQNi,0.000179,7.311335e-05,0.253171,0.000058,2020-12-31
25ErA4HzoCTi1KTHh5b7MDBjdxEPuGbDcuCqz931P4nn,0.000179,4.280621e-05,0.385612,0.000047,2020-12-31
GsMzKMTsEdWAztitmoKGxCb58CpKkGehQ26H2FmTJvpQ,0.000179,0.000000e+00,0.385584,0.000047,2020-12-31
CoNug4D8ZP57NB7RZfvh7pGM1NNjb32PZAnZXREKJ9rB,0.000179,7.311335e-05,0.253171,0.000058,2020-12-31


In [ ]:
output_path = RESULTS_DIR / "solana" / "network_metrics" / "SOL_2020-10-01_to_2020-12-31_centrality.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output_path)
